In [9]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [ ]:
load_dotenv(override=True)

In [ ]:
# Step 1:: Define the State Object

class State(BaseModel):
    messages: Annotated[list, add_messages]

In [ ]:
# Step 2:: Start the Graph Builder with the State class

graph_builder = StateGraph(State)

In [ ]:
# Step 3:: Create a Node

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot_node(old_state: State) -> State:
    response = llm.invoke(old_state.messages)
    new_state = State(messages=[response])
    return new_state

graph_builder.add_node("chatbot", chatbot_node)

In [ ]:
# Step 4:: Create Edges

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

In [ ]:
# Step 5:: Compile the Graph

graph = graph_builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
def chat(user_input: str, history):
    initial_state = State(messages=[{"role": "user", "content": user_input}])

    result = graph.invoke(initial_state)
    print(result)

    return result['messages'][-1].content

gr.ChatInterface(chat).launch()